In [1]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import scanpy.external as sce
import anndata as an
import scipy
import sklearn
import gget

sc.settings.verbosity = 3  

In [2]:
fpath = "/scratch/indikar_root/indikar1/shared_data/cc_dynamics/sc_fib.h5ad"

bdata = sc.read_h5ad(fpath)
bdata.var_names = bdata.var['gene_name'].values
bdata.obs['cell'] = bdata.obs.index.copy()
sc.logging.print_memory_usage()

dpt_map = dict(zip(bdata.obs['cell'].values, bdata.obs['dpt_pseudotime'].values))

bdata

Memory usage: current 1.24 GB, difference +1.24 GB


AnnData object with n_obs × n_vars = 7082 × 4731
    obs: 'n_genes', 'cell', 'G1', 'G2M', 'S', 'pred_phase', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'leiden', 'pred_G1', 'pred_S', 'pred_G2M', 'dpt_pseudotime'
    var: 'gene_name'
    layers: 'zscore'

In [3]:
fpath = "/scratch/indikar_root/indikar1/shared_data/single_cell_fibroblast/scanpy/processed.anndata.h5ad"

adata = sc.read_h5ad(fpath)
adata.obs['cell'] = adata.obs.index.copy()
sc.logging.print_memory_usage()

adata

Memory usage: current 3.30 GB, difference +2.05 GB


AnnData object with n_obs × n_vars = 7748 × 14082
    obs: 'n_genes', 'cell', 'G1', 'G2M', 'S', 'pred_phase', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'leiden', 'pred_G1', 'pred_S', 'pred_G2M', 'dpt_pseudotime'
    var: 'gene_name', 'Chromosome', 'Start', 'End', 'Strand', 'seurat_S', 'seurat_G2M', 'is_seurat', 'is_kegg', 'whitfield_G1/S', 'whitfield_G2', 'whitfield_G2/M', 'whitfield_M/G1', 'whitfield_S', 'is_whitfield', 'GO_G1', 'GO_G1/S', 'GO_G2', 'GO_G2/M', 'GO_M', 'GO_S', 'is_GO', 'LIU_G2MvsG0G1', 'LIU_G2MvsS', 'LIU_SvsG0G1', 'is_LIU', 'cell_cycle', 'n_cells', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_count

In [4]:
# map the DPT values
adata.obs['dpt'] = adata.obs['cell'].map(dpt_map)

# drop NAN dpt values
adata = adata[adata.obs['dpt'].notna(), :].copy()

adata.obs['dpt_hour'] = pd.cut(
    adata.obs['dpt'], 
    bins=24, 
    labels=False, 
    right=False, 
    include_lowest=True,
)

adata.obs['dpt_hour'].value_counts()

dpt_hour
18    1406
1     1308
19    1117
2      944
0      810
17     675
20     275
16     133
3       91
4       54
15      53
5       30
14      30
6       27
13      22
12      21
7       19
8       17
9       14
11      13
21      11
10       8
23       1
22       1
Name: count, dtype: int64

In [23]:
mask = (adata.obs['dpt_hour'] < 10) & (adata.obs['pred_phase'] == 'G1')
g1 = adata[mask, :].copy()
g1.X = g1.layers['counts'].copy()
g1.raw = g1
g1

AnnData object with n_obs × n_vars = 3012 × 14082
    obs: 'n_genes', 'cell', 'G1', 'G2M', 'S', 'pred_phase', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'leiden', 'pred_G1', 'pred_S', 'pred_G2M', 'dpt_pseudotime', 'dpt', 'dpt_hour'
    var: 'gene_name', 'Chromosome', 'Start', 'End', 'Strand', 'seurat_S', 'seurat_G2M', 'is_seurat', 'is_kegg', 'whitfield_G1/S', 'whitfield_G2', 'whitfield_G2/M', 'whitfield_M/G1', 'whitfield_S', 'is_whitfield', 'GO_G1', 'GO_G1/S', 'GO_G2', 'GO_G2/M', 'GO_M', 'GO_S', 'is_GO', 'LIU_G2MvsG0G1', 'LIU_G2MvsS', 'LIU_SvsG0G1', 'is_LIU', 'cell_cycle', 'n_cells', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts'

# MAGIC

In [25]:
sce.pp.magic(
    g1, 
    name_list='all_genes', 
    solver='exact',
    knn=5,
)

g1.layers['magic'] = g1.X.copy()
g1

computing MAGIC
  Running MAGIC with `solver='exact'` on 14082-dimensional data may take a long time. Consider denoising specific genes with `genes=<list-like>` or using `solver='approximate'`.
    finished (0:00:12)


AnnData object with n_obs × n_vars = 3012 × 14082
    obs: 'n_genes', 'cell', 'G1', 'G2M', 'S', 'pred_phase', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'leiden', 'pred_G1', 'pred_S', 'pred_G2M', 'dpt_pseudotime', 'dpt', 'dpt_hour'
    var: 'gene_name', 'Chromosome', 'Start', 'End', 'Strand', 'seurat_S', 'seurat_G2M', 'is_seurat', 'is_kegg', 'whitfield_G1/S', 'whitfield_G2', 'whitfield_G2/M', 'whitfield_M/G1', 'whitfield_S', 'is_whitfield', 'GO_G1', 'GO_G1/S', 'GO_G2', 'GO_G2/M', 'GO_M', 'GO_S', 'is_GO', 'LIU_G2MvsG0G1', 'LIU_G2MvsS', 'LIU_SvsG0G1', 'is_LIU', 'cell_cycle', 'n_cells', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts'

In [28]:
matrix = g1.layers['magic']

# Count negative values
num_negative = np.sum(matrix < 0)

# Calculate total number of values
total_values = matrix.size

# Calculate percentage
percentage_negative = (num_negative / total_values) * 100

# --- Descriptive Statistics ---
print("Matrix Descriptive Statistics:")
print(f"  Minimum value: {np.min(matrix)}")
print(f"  Maximum value: {np.max(matrix)}")
print(f"  Mean value: {np.mean(matrix):.2f}")
print(f"  Standard deviation: {np.std(matrix):.2f}")

# Print the results
print("\nNegative Value Analysis:")
print(f"  Number of negative values: {num_negative}")
print(f"  Percentage of negative values: {percentage_negative:.2f}%")

Matrix Descriptive Statistics:
  Minimum value: 0.0
  Maximum value: 495.1466289432671
  Mean value: 0.66
  Standard deviation: 4.38

Negative Value Analysis:
  Number of negative values: 0
  Percentage of negative values: 0.00%


# Save data

In [36]:
outpath = "../../../temp/g1_magic_imputed.csv"
aggdata = sc.get.aggregate(
    g1,
    by='dpt_hour',
    func='mean',
    layer='magic'
)

df = aggdata.to_df(layer='mean')
df = df.reset_index(drop=False, names='HOUR')
df['HOUR'] = df['HOUR'].astype(int) + 1
print(f"{df.shape=}")
df.to_csv(outpath)
df.head()

df.shape=(10, 14083)


,HOUR,ATAD3B,SKI,PEX14,PLCH2,HES3,PLEKHM2,CA6,NMNAT1,CCDC27,...,TSPY4,TSPY9,KDM5D,RBMY1F,BPY2C,CDY2B,SRY,VCY,DAZ1,RBMY1E
0,1,0.133924,0.432464,0.047734,0.023633,0.044747,0.620529,0.047804,0.048983,0.110142,...,0.011759,0.300643,0.023989,0.000662,0.002579,0.001374,0.003390,0.130997,0.001144,0.037020
1,2,0.137856,0.439082,0.048251,0.022669,0.045091,0.631181,0.046018,0.052356,0.117260,...,0.012802,0.314788,0.026563,0.001373,0.002359,0.001495,0.004404,0.134263,0.001448,0.033676
2,3,0.149216,0.449280,0.050803,0.021313,0.047831,0.650798,0.046121,0.054322,0.128628,...,0.013972,0.343590,0.033104,0.002770,0.001913,0.001730,0.005704,0.145888,0.001631,0.029600
3,4,0.138173,0.416139,0.050303,0.021959,0.045847,0.628787,0.049534,0.052764,0.109080,...,0.012590,0.327970,0.035793,0.002976,0.001749,0.001506,0.004731,0.139686,0.001116,0.031087
4,5,0.140915,0.433492,0.048922,0.021212,0.049120,0.649322,0.052066,0.050163,0.111966,...,0.012356,0.339711,0.037883,0.006105,0.001758,0.001520,0.005249,0.156741,0.001959,0.022483
